After any interrupted run, restart the runtime before re-running this notebook. Re-running %pip install in a live session produces exactly this stale-import failure.


# Phase 6 v2: frozen benchmark evaluation on Kaggle

This evaluation-only notebook compares the pinned base model and saved Phase 6 adapter on the frozen 60-case benchmark. Upload one Kaggle dataset containing the adapter directory contents and `phase6-v2-base-report.json`. Kaggle requires phone verification to enable Internet; turn Internet ON and select a GPU accelerator before running. Reports are written to `/kaggle/working/evaluation/`.


Run this notebook with Run All while configuring it, not Save & Run All, because a committed run executes headlessly and you see nothing until it finishes.


In [ ]:
import socket
import subprocess
from pathlib import Path

ADAPTER_DIR_OVERRIDE = None
BASE_REPORT_PATH_OVERRIDE = None
KAGGLE_INPUT_ROOT = Path("/kaggle/input")
failures = []
warnings = []
adapter_candidates = []
base_report_candidates = []
adapter_path = None
base_report_path = None
gpu_name = None

if not KAGGLE_INPUT_ROOT.is_dir():
    failures.append(
        "Inputs: /kaggle/input does not exist; attach the dataset via "
        "+ Add Input in the notebook sidebar."
    )
else:
    input_entries = list(KAGGLE_INPUT_ROOT.iterdir())
    if not input_entries:
        failures.append(
            "Inputs: /kaggle/input is empty; attach the dataset via "
            "+ Add Input in the notebook sidebar."
        )
    adapter_candidates = sorted(
        {
            path.parent
            for path in KAGGLE_INPUT_ROOT.rglob("adapter-metadata.json")
            if path.is_file()
        }
    )
    base_report_candidates = sorted(
        path
        for path in KAGGLE_INPUT_ROOT.rglob("phase6-v2-base-report.json")
        if path.is_file()
    )

if ADAPTER_DIR_OVERRIDE is not None:
    adapter_path = Path(ADAPTER_DIR_OVERRIDE)
    if not (
        adapter_path.is_dir() and (adapter_path / "adapter-metadata.json").is_file()
    ):
        failures.append(
            f"Adapter override is not a directory containing adapter-metadata.json: "
            f"{adapter_path}."
        )
elif len(adapter_candidates) != 1:
    failures.append(
        f"Adapter discoverability: found {len(adapter_candidates)} directories "
        "containing adapter-metadata.json; "
        "set ADAPTER_DIR_OVERRIDE to the intended directory."
    )
else:
    adapter_path = adapter_candidates[0]

if BASE_REPORT_PATH_OVERRIDE is not None:
    base_report_path = Path(BASE_REPORT_PATH_OVERRIDE)
    if not base_report_path.is_file():
        failures.append(
            f"Base report override does not name a file: {base_report_path}."
        )
elif len(base_report_candidates) == 1:
    base_report_path = base_report_candidates[0]
elif len(base_report_candidates) > 1:
    failures.append(
        f"Base report discoverability: found {len(base_report_candidates)} "
        "phase6-v2-base-report.json files; "
        "set BASE_REPORT_PATH_OVERRIDE to the intended report."
    )
else:
    warnings.append(
        "Base report: No base report found under /kaggle/input; the notebook "
        "will regenerate it, which will cost roughly double the GPU time."
    )

try:
    with socket.create_connection(("pypi.org", 443), timeout=5):
        pass
except OSError as error:
    failures.append(
        "Internet: pypi.org is unreachable; enable Internet in the notebook sidebar, "
        f"which requires phone verification at kaggle.com/settings. ({error})"
    )

try:
    gpu_probe = subprocess.run(
        ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
        capture_output=True,
        text=True,
        check=False,
        timeout=3,
    )
    gpu_name = (
        gpu_probe.stdout.strip().splitlines()[0]
        if gpu_probe.returncode == 0 and gpu_probe.stdout.strip()
        else None
    )
except (FileNotFoundError, OSError, subprocess.TimeoutExpired):
    gpu_name = None
if gpu_name is None:
    try:
        gpu_version = (
            Path("/proc/driver/nvidia/version").read_text(encoding="utf-8").strip()
        )
        gpu_name = gpu_version.splitlines()[0] if gpu_version else None
    except (OSError, UnicodeDecodeError):
        gpu_name = None
if gpu_name is None:
    failures.append(
        "GPU: no NVIDIA device found; set Accelerator to GPU T4 x2 in the sidebar."
    )

if warnings:
    for warning in warnings:
        print(f"WARNING: {warning}")
if failures:
    raise RuntimeError("Preflight failed:\n- " + "\n- ".join(failures))
print(
    "Preflight passed:",
    f"adapter={adapter_path}, base_report={base_report_path or 'will regenerate'}, "
    f"GPU={gpu_name}",
)

In [ ]:
%pip install -q transformers==5.14.1 accelerate==1.14.0
%pip install -q bitsandbytes==0.50.0 peft==0.20.0 safetensors==0.8.0

import shutil
import subprocess
import sys
from pathlib import Path

KAGGLE_INPUT_ROOT = Path("/kaggle/input")
KAGGLE_WORKING_ROOT = Path("/kaggle/working")
EVALUATION_DIR = KAGGLE_WORKING_ROOT / "evaluation"


def _input_files(filename):
    if not KAGGLE_INPUT_ROOT.is_dir():
        return []
    return sorted(
        path
        for dataset_root in KAGGLE_INPUT_ROOT.iterdir()
        if dataset_root.is_dir()
        for path in dataset_root.rglob(filename)
        if path.is_file()
    )


def resolve_adapter_dir():
    if ADAPTER_DIR_OVERRIDE is not None:
        path = Path(ADAPTER_DIR_OVERRIDE)
        if not path.is_dir():
            raise RuntimeError(
                f"ADAPTER_DIR_OVERRIDE does not name a directory: {path}"
            )
        return path
    metadata_paths = _input_files("adapter-metadata.json")
    candidates = sorted({path.parent for path in metadata_paths})
    if not candidates:
        raise RuntimeError(
            "No adapter metadata found under /kaggle/input. Found: none. "
            "Set ADAPTER_DIR_OVERRIDE to the adapter directory."
        )
    if len(candidates) != 1:
        raise RuntimeError(
            "Ambiguous adapter discovery under /kaggle/input: "
            f"{[str(path) for path in candidates]}. "
            "Set ADAPTER_DIR_OVERRIDE to the intended adapter directory."
        )
    return candidates[0]


def resolve_base_report_path():
    if BASE_REPORT_PATH_OVERRIDE is not None:
        return Path(BASE_REPORT_PATH_OVERRIDE)
    candidates = _input_files("phase6-v2-base-report.json")
    if not candidates:
        return None
    if len(candidates) != 1:
        raise RuntimeError(
            "Ambiguous base report discovery under /kaggle/input: "
            f"{[str(path) for path in candidates]}. "
            "Set BASE_REPORT_PATH_OVERRIDE to the intended report path."
        )
    return candidates[0]


ADAPTER_DIR = resolve_adapter_dir()
BASE_REPORT_PATH = resolve_base_report_path()
print({"resolved_adapter_directory": str(ADAPTER_DIR)})
print({"resolved_base_report_path": str(BASE_REPORT_PATH)})

REPOSITORY_URL = "https://github.com/muzzary/GTM-Agent.git"
REPOSITORY_REF = "codex/phase-6-reviewed-adapter"
REPOSITORY_DIR = KAGGLE_WORKING_ROOT / "GTM-Agent"
if REPOSITORY_DIR.exists():
    subprocess.run(
        ["git", "-C", str(REPOSITORY_DIR), "fetch", "origin", REPOSITORY_REF],
        check=True,
    )
    subprocess.run(
        ["git", "-C", str(REPOSITORY_DIR), "checkout", REPOSITORY_REF],
        check=True,
    )
    subprocess.run(
        ["git", "-C", str(REPOSITORY_DIR), "pull", "--ff-only"],
        check=True,
    )
else:
    subprocess.run(
        [
            "git",
            "clone",
            "--depth",
            "1",
            "--branch",
            REPOSITORY_REF,
            REPOSITORY_URL,
            str(REPOSITORY_DIR),
        ],
        check=True,
    )
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", str(REPOSITORY_DIR)],
    check=True,
)
sys.path.insert(0, str(REPOSITORY_DIR))

In [ ]:
import importlib
import importlib.metadata

PINNED_VERSIONS = {
    "transformers": "5.14.1",
    "accelerate": "1.14.0",
    "bitsandbytes": "0.50.0",
    "peft": "0.20.0",
    "safetensors": "0.8.0",
}
RESTART_INSTRUCTION = (
    "Runtime > Restart session, then run all cells top to bottom. "
    "Do not re-run the install cell in a live session."
)
packages_without_version = []
for package_name, pinned_version in PINNED_VERSIONS.items():
    installed_version = importlib.metadata.version(package_name)
    imported_version = getattr(
        importlib.import_module(package_name), "__version__", None
    )
    if installed_version != pinned_version or (
        imported_version is not None and imported_version != installed_version
    ):
        raise RuntimeError(
            f"{package_name} version mismatch: installed={installed_version}, "
            f"imported={imported_version}. {RESTART_INSTRUCTION}"
        )
    if imported_version is None:
        packages_without_version.append(package_name)
print({"packages_without_version": packages_without_version})

In [ ]:
import gc
import json
import time
from datetime import UTC, datetime
from hashlib import sha256

import torch
from peft import PeftModel
from pydantic import ValidationError
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

from src.evaluation.phase1 import load_manifest as load_phase1_manifest
from src.evaluation.phase6_benchmark import (
    audit_phase6_benchmark,
    load_phase6_benchmark,
)
from src.evaluation.phase6_v2 import (
    Phase6V2EvaluationReport,
    build_phase6_v2_prompt,
    compare_phase6_v2_reports,
    run_phase6_v2_evaluation,
)
from src.schemas.dataset import DatasetManifest, DatasetManifestV2
from src.schemas.inference import (
    GenerationSettings,
    GroundedOutreachOutput,
    ModelIdentity,
)
from src.schemas.training import AdapterArtifactMetadata, TrainingConfigV2

CONFIG_PATH = REPOSITORY_DIR / "configs/phase6/training-v2.json"
PILOT_PATH = REPOSITORY_DIR / "configs/phase6/pilot.json"
PHASE1_PATH = REPOSITORY_DIR / "configs/phase1/benchmark.json"
V2_DATASET_PATH = REPOSITORY_DIR / "configs/phase6/dataset-v2.json"
V2_BENCHMARK_PATH = REPOSITORY_DIR / "configs/phase6/benchmark-v2.json"
config = TrainingConfigV2.model_validate_json(CONFIG_PATH.read_text(encoding="utf-8"))
pilot = DatasetManifest.model_validate_json(PILOT_PATH.read_text(encoding="utf-8"))
v2_dataset = DatasetManifestV2.model_validate_json(
    V2_DATASET_PATH.read_text(encoding="utf-8")
)
phase1 = load_phase1_manifest(PHASE1_PATH)
v2_benchmark = load_phase6_benchmark(V2_BENCHMARK_PATH)
assert config.base_model_id == "Qwen/Qwen3-4B-Instruct-2507"
assert config.base_model_revision == "cdbee75f17c01a7cc42f958dc650907174af0554"
blocked_identities = {
    *(f"product:{item.product_group}" for item in pilot.examples),
    *(f"icp:{item.icp_group}" for item in pilot.examples),
    *(f"company:{item.company_group}" for item in pilot.examples),
    *(f"prospect:{item.prospect_group}" for item in pilot.examples),
}
benchmark_audit = audit_phase6_benchmark(
    v2_benchmark,
    blocked_identity_groups=blocked_identities,
    blocked_case_ids={case.case_id for case in phase1.cases},
)
assert benchmark_audit.evaluation_ready
assert benchmark_audit.total_cases == 60
assert torch.cuda.is_available(), "Select a Kaggle GPU accelerator before evaluation."

ADAPTER_WORKING_DIR = KAGGLE_WORKING_ROOT / "adapter"
shutil.copytree(ADAPTER_DIR, ADAPTER_WORKING_DIR, dirs_exist_ok=True)
required_adapter_files = (
    "adapter_model.safetensors",
    "adapter_config.json",
    "adapter-metadata.json",
)
missing_adapter_files = [
    name
    for name in required_adapter_files
    if not (ADAPTER_WORKING_DIR / name).is_file()
]
if missing_adapter_files:
    raise RuntimeError(
        f"Writable adapter copy is incomplete at {ADAPTER_WORKING_DIR}: "
        f"missing {missing_adapter_files}"
    )
probe_path = ADAPTER_WORKING_DIR / ".write-probe"
probe_path.write_text("verified", encoding="utf-8")
probe_path.unlink()
print({"writable_adapter_copy": str(ADAPTER_WORKING_DIR)})

metadata = AdapterArtifactMetadata.model_validate_json(
    (ADAPTER_WORKING_DIR / "adapter-metadata.json").read_text(encoding="utf-8")
)
assert metadata.adapter_id == config.adapter_id
assert metadata.dataset_id == v2_dataset.dataset_id
assert metadata.dataset_version == v2_dataset.dataset_version
assert metadata.base_model_id == config.base_model_id
assert metadata.base_model_revision == config.base_model_revision
print({"epochs_completed": metadata.epochs_completed})

In [ ]:
quantization = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)
BASE_IDENTITY = ModelIdentity(
    model_id=config.base_model_id, model_revision=config.base_model_revision
)
ADAPTER_IDENTITY = ModelIdentity(
    model_id=config.base_model_id,
    model_revision=config.base_model_revision,
    adapter_id=metadata.adapter_id,
    adapter_revision=metadata.adapter_revision,
)


class ModelOutputError(ValueError):
    def __init__(self, message, raw_output):
        super().__init__(message)
        self.raw_output_excerpt = raw_output[:2000]


def load_evaluation_model(adapter_dir=None):
    tokenizer = AutoTokenizer.from_pretrained(
        config.base_model_id,
        revision=config.base_model_revision,
        trust_remote_code=False,
        use_fast=True,
    )
    model = AutoModelForCausalLM.from_pretrained(
        config.base_model_id,
        revision=config.base_model_revision,
        quantization_config=quantization,
        device_map="auto",
        trust_remote_code=False,
        use_safetensors=True,
    )
    if adapter_dir is not None:
        model = PeftModel.from_pretrained(model, adapter_dir, is_trainable=False)
    model.eval()
    return tokenizer, model


def generate_grounded_output(request, tokenizer, model):
    messages = [
        {"role": "system", "content": "Return strict JSON only."},
        {"role": "user", "content": request.prompt},
    ]
    encoded = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    ).to(model.device)
    torch.manual_seed(request.seed)
    torch.cuda.manual_seed_all(request.seed)
    started = time.perf_counter()
    with torch.inference_mode():
        output_ids = model.generate(
            **encoded,
            do_sample=False,
            max_new_tokens=request.max_new_tokens,
            pad_token_id=tokenizer.eos_token_id,
        )
    generated = output_ids[0, encoded["input_ids"].shape[-1] :]
    raw_output = tokenizer.decode(generated, skip_special_tokens=True).strip()
    try:
        result = GroundedOutreachOutput.model_validate(json.loads(raw_output))
    except (json.JSONDecodeError, ValidationError) as error:
        raise ModelOutputError(
            f"model output does not match the v2 contract: {error}", raw_output
        ) from error
    print(
        {
            "request_id": request.request_id,
            "latency_seconds": round(time.perf_counter() - started, 2),
            "status": result.generation_status,
        }
    )
    return result


GENERATION_SETTINGS = GenerationSettings(max_new_tokens=768, seed=42)


def evaluate_model(tokenizer, model, identity):
    return run_phase6_v2_evaluation(
        v2_benchmark,
        identity,
        lambda request: generate_grounded_output(request, tokenizer, model),
        max_new_tokens=GENERATION_SETTINGS.max_new_tokens,
        seed=GENERATION_SETTINGS.seed,
    )


RUN_ID = datetime.now(UTC).strftime("%Y%m%dT%H%M%SZ")
EVALUATION_DIR.mkdir(parents=True, exist_ok=True)
print({"evaluation_directory": str(EVALUATION_DIR)})

In [ ]:
BASE_REPORT_DESTINATION = EVALUATION_DIR / "phase6-v2-base-report.json"


def regenerate_base_report():
    base_tokenizer, base_model = load_evaluation_model()
    report = evaluate_model(base_tokenizer, base_model, BASE_IDENTITY)
    BASE_REPORT_DESTINATION.write_text(
        report.model_dump_json(indent=2), encoding="utf-8"
    )
    del base_model, base_tokenizer
    gc.collect()
    torch.cuda.empty_cache()
    return report


if BASE_REPORT_PATH is not None and BASE_REPORT_PATH.exists():
    base_report = Phase6V2EvaluationReport.model_validate_json(
        BASE_REPORT_PATH.read_text(encoding="utf-8")
    )
    assert (
        base_report.benchmark_id == v2_benchmark.benchmark_id
        and base_report.benchmark_manifest_sha256 == v2_benchmark.content_sha256
    ), "reused base report benchmark identity mismatch"
    assert base_report.generation == GENERATION_SETTINGS, (
        "reused base report generation settings mismatch"
    )
    assert (
        base_report.model.model_id == BASE_IDENTITY.model_id
        and base_report.model.model_revision == BASE_IDENTITY.model_revision
        and base_report.model.adapter_id is None
        and base_report.model.adapter_revision is None
    ), "reused base report model identity mismatch"
    assert [case.case_id for case in base_report.cases] == [
        case.case_id for case in v2_benchmark.cases
    ], "reused base report case IDs mismatch"
    assert base_report.total_cases == len(v2_benchmark.cases), (
        "reused base report total_cases mismatch"
    )
    for benchmark_case, report_case in zip(
        v2_benchmark.cases, base_report.cases, strict=True
    ):
        current_prompt_sha256 = sha256(
            build_phase6_v2_prompt(benchmark_case).encode("utf-8")
        ).hexdigest()
        assert report_case.prompt_sha256 == current_prompt_sha256, (
            f"reused base report prompt hash mismatch for {benchmark_case.case_id}: "
            f"report={report_case.prompt_sha256}, current={current_prompt_sha256}"
        )
    BASE_REPORT_DESTINATION.write_text(
        base_report.model_dump_json(indent=2), encoding="utf-8"
    )
    print({"base_report_mode": "reused", "base_report_source": str(BASE_REPORT_PATH)})
else:
    print({"base_report_mode": "regenerated"})
    base_report = regenerate_base_report()
print(
    {
        "base_report": str(BASE_REPORT_DESTINATION),
        "valid_outputs": base_report.valid_output_count,
        "deterministic_passes": base_report.deterministic_passed_case_count,
    }
)

In [ ]:
adapter_tokenizer, adapter_model = load_evaluation_model(ADAPTER_WORKING_DIR)
adapter_report = evaluate_model(adapter_tokenizer, adapter_model, ADAPTER_IDENTITY)
ADAPTER_REPORT_PATH = EVALUATION_DIR / "phase6-v2-adapter-report.json"
ADAPTER_REPORT_PATH.write_text(
    adapter_report.model_dump_json(indent=2), encoding="utf-8"
)
comparison = compare_phase6_v2_reports(base_report, adapter_report)
COMPARISON_PATH = EVALUATION_DIR / "phase6-v2-comparison.json"
COMPARISON_PATH.write_text(comparison.model_dump_json(indent=2), encoding="utf-8")
print(comparison.model_dump(mode="json"))
print(
    {
        "evaluation_directory": str(EVALUATION_DIR),
        "base_report": str(BASE_REPORT_DESTINATION),
        "adapter_report": str(ADAPTER_REPORT_PATH),
        "comparison": str(COMPARISON_PATH),
    }
)

## Reusing or regenerating the base report

With `BASE_REPORT_PATH_OVERRIDE = None`, the notebook recursively discovers exactly one `phase6-v2-base-report.json` under `/kaggle/input` and reuses it only after five fail-closed assertions. If the report is absent, set `BASE_REPORT_PATH_OVERRIDE` to a writable path such as `/kaggle/working/evaluation/phase6-v2-base-report.json` and re-run from the top; the explicit regeneration branch downloads the pinned base model and writes that report before adapter evaluation.

The adapter is copied to `/kaggle/working/adapter`, and a write probe verifies that PEFT receives a writable path rather than the read-only Kaggle input mount.


## Reading the result

`inconclusive` means the adapter missed the 95% valid-output threshold or at least one deterministic case gate. `pending_semantic_review` means the outputs are structurally ready for blind human scoring; it is not acceptance.
